# Lab 23-09 · An Analog Matrix-Vector Multiply

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import cim


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · ngspice

*Predict* the column currents, then whether resistive sensing raises or lowers the outputs.

In [2]:
sim = ngspice("crossbar.cir")
for k in ("i1p", "i1n", "i2p", "i2n", "y1", "y2", "y1b", "y2b"):
    print(f"{k:4s} = {sim[k]*1e6:8.3f} uA")
W = np.array([[1, -2], [3, 0], [-1, 2], [2, 1]])
print("exact products:", np.array([2, 1, 0, 3]) @ W)

i1p  =   11.600 uA
i1n  =    0.600 uA
i2p  =    3.600 uA
i2n  =    4.600 uA
y1   =   11.000 uA
y2   =   -1.000 uA
y1b  =    6.547 uA
y2b  =   -1.023 uA
exact products: [11 -1]


## Part 2 · Compute SNR

*Predict* the SNR for 5% variation.

In [3]:
rng = np.random.default_rng(2309)
rows, cols, sigma = 256, 64, 0.05
w = rng.normal(size=(rows, cols))
snr = []
for _ in range(200):
    x = rng.normal(size=rows)
    ideal = x @ w
    noisy = x @ (w * (1 + sigma * rng.normal(size=w.shape)))
    snr.append(np.mean(ideal ** 2) / np.mean((noisy - ideal) ** 2))
print(f"measured SNR {10*np.log10(np.mean(snr)):.2f} dB; model {10*np.log10(cim.snr_from_sigma(sigma)):.2f} dB")

measured SNR 26.17 dB; model 26.02 dB


## Part 3 · The ADC ceiling

*Predict* how the ceiling scales with precision.

In [4]:
bits = np.arange(1, 9)
plt.figure()
for fom in (1e-15, 5e-15, 20e-15):
    plt.semilogy(bits, [cim.tops_per_watt(cim.adc_energy_per_mac(fom, 8, 256, b, b)) for b in bits], "o-",
                 label=f"FoM {fom*1e15:.0f} fJ/step")
plt.xlabel("input and weight bits"); plt.ylabel("ADC-limited TOPS/W"); plt.legend(); plt.show()
print(f"8-bit ceiling at 5 fJ/step: {cim.tops_per_watt(cim.adc_energy_per_mac(5e-15, 8, 256, 8, 8)):.2f} TOPS/W")

8-bit ceiling at 5 fJ/step: 6.25 TOPS/W
